In [ ]:
%pip install -q groq

Note: you may need to restart the kernel to use updated packages.


In [1]:
import json
import os
from pathlib import Path

from groq import Groq

In [2]:
MODEL = "llama-3.1-8b-instant"

In [3]:

def read_json(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError as e:
        raise FileNotFoundError(f"JSON file not found: {path}") from e
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON in file: {path}") from e


def read_text(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError as e:
        raise FileNotFoundError(f"Text file not found: {path}") from e


def generate_text(prompt_template, input_data):
    api_key = os.getenv("GROQ_API")
    if not api_key:
        raise EnvironmentError("Environment variable GROQ_API_KEY is not set")

    client = Groq(api_key=api_key)

    user_prompt = f"Данные пользователя:\n{json.dumps(input_data, ensure_ascii=False, indent=2)}"
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": f"{prompt_template.strip()}"},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0.7,
        )
        text = response.choices[0].message.content
        if not text or not text.strip():
            raise ValueError("LLM returned empty text")
        return text.strip()
    except Exception as e:
        raise RuntimeError(f"Groq API call failed: {e}") from e


def save_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

In [4]:
input_path = Path("input/input.json")
prompt_path = Path("prompts/summary.txt")
output_path = Path("output/summary.json")

In [6]:
try:
    input_data = read_json(input_path)
    prompt_template = read_text(prompt_path)
    
    generated_text = generate_text(prompt_template, input_data)
    
    save_json(
        output_path,
        {
            "field": "summary",
            "text": generated_text,
        },
    )
    print(f"Saved: {output_path}")
    print("Pipeline finished successfully.")

except Exception as e:
    print(f"Pipeline failed: {e}")

if output_path.exists():
    print(f"\n{output_path.name}:")
    print(read_text(output_path))
else:
    print(f"Missing: {output_path}")

Saved: output/summary.json
Pipeline finished successfully.

summary.json:
{
  "field": "summary",
  "text": "Я - перспективный студент-разработчик, нацеленный на карьеру Junior Ml Developer. На втором курсе бакалавриата по Computer Science, я уже приобрел опыт в работе с технологиями машинного обучения и глубокого обучения, в том числе с использованием Python, FastAPI, PostgreSQL, Docker и Git. Мне особенно интересен опыт в проектах, связанных с машинным обучением и анализом данных, таких как время серии модели для прогноза лучших рекламных каналов, с которым я добился успеха в хакатонах METU и IT-Fest."
}
